# Week 2 — GROUP BY, Aggregates, HAVING: Review Score Analysis
## Phase 2b SQL | PORA Academy Cohort 7 — **Exercises**

On Wednesday you grouped orders, payments and customers. Today you point the same tools at the `order_reviews` table and ask a harder question: **are Olist's customers happy?** A single average can hide a lot — part of today's work is seeing the *shape* of the score distribution, not just its mean.

Each question below comes as **three cells**:

1. A **question** with the task and an **Expected** result.
2. A blank `%%sql` answer cell — write your query where it says `-- Your query here`, capturing the result into a variable (e.g. `q1`).
3. A **check cell** (plain Python) — run it after your query. A ✅ means you got it right, and the cell then displays the table your query returned.

**Do not edit the check cells.** Run the setup cell first, then work top to bottom.

Reminders for this week:
- Every non-aggregated column in the `SELECT` list must also appear in `GROUP BY`.
- `WHERE` filters **rows before** grouping; `HAVING` filters **groups after** grouping.
- SQLite does integer division: `COUNT(*) / total` truncates to 0. Force a real number with `* 100.0`.
- Alias your aggregates exactly as each question asks — the check cells look up those column names.

The last question is a **discussion** question with no answer cell — read it, but don't try to write SQL for it yet.

In [11]:
# =====================================================================
# Olist SQL Setup — runs on BOTH Google Colab and a local machine.
# Run this cell FIRST. It loads the 8 Olist tables into a SQLite
# database and connects the %%sql magic to it. You should not need to
# edit anything unless auto-detection fails (see the two knobs below).
#
# Design notes:
# - We teach SQL with the %%sql cell magic (jupysql), not pd.read_sql().
# - jupysql opens its OWN connection, so the DB must be a real FILE
#   (a :memory: DB would be invisible to it).
# - We use jupysql (the maintained SQL magic). On Colab we install it,
#   because Colab ships the legacy ipython-sql, which (a) can't take a
#   connection by engine variable and (b) renders every result through
#   prettytable.__dict__[style], crashing on modern prettytable with
#   KeyError 'DEFAULT'/'SINGLE_BORDER'. jupysql fixes both.
# - autopandas=True makes every %%sql result a pandas DataFrame, which
#   lets the self-check cells assert on .iloc/.shape directly.
# =====================================================================
import os, glob, sqlite3, tempfile, zipfile
import pandas as pd

# --- Optional knobs (leave blank; only set if auto-detect fails) ------
LOCAL_DATA_DIR = ""   # local run: folder that holds olist_orders_dataset.csv
DRIVE_ZIP_PATH = ""   # Colab: full path to phase-2-python-sql.zip in your Drive
# ---------------------------------------------------------------------

# Detect Colab (google.colab only imports there). Outside Colab — including
# the content-pipeline validator — this falls through to the local branch.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except ModuleNotFoundError:
    ON_COLAB = False


def _colab_find_zip():
    """Locate phase-2-python-sql.zip in Drive WITHOUT a full recursive scan
    (globbing '/content/drive/MyDrive/**' walks the entire Drive over the
    network and can hang for many minutes). Try explicit paths first, then a
    depth- and count-bounded breadth-first search that prints progress."""
    if DRIVE_ZIP_PATH:
        if os.path.exists(DRIVE_ZIP_PATH):
            return DRIVE_ZIP_PATH
        raise FileNotFoundError(f"DRIVE_ZIP_PATH is set but not found: {DRIVE_ZIP_PATH}")

    target = "phase-2-python-sql.zip"
    # Fast, instant checks of the most likely spots (top of Drive + course folder).
    for cand in (
        f"/content/drive/MyDrive/{target}",
        f"/content/drive/MyDrive/Data Analysis and AI Automation Course Cohort 7/Dataset/{target}",
        f"/content/{target}",
    ):
        if os.path.exists(cand):
            return cand

    # Bounded BFS: depth <= 4, at most ~600 folders, skipping hidden dirs.
    print("Searching your Google Drive for phase-2-python-sql.zip ...")
    root, queue, scanned = "/content/drive/MyDrive", [("/content/drive/MyDrive", 0)], 0
    while queue:
        d, depth = queue.pop(0)
        hit = os.path.join(d, target)
        if os.path.exists(hit):
            return hit
        if depth >= 4:
            continue
        try:
            for e in os.scandir(d):
                if e.is_dir() and not e.name.startswith("."):
                    queue.append((e.path, depth + 1))
        except OSError:
            continue
        scanned += 1
        if scanned % 50 == 0:
            print(f"  ...scanned {scanned} folders")
        if scanned >= 600:
            break

    raise FileNotFoundError(
        "Could not quickly find phase-2-python-sql.zip in your Drive. Put the zip at the "
        "TOP of your Drive (My Drive) and re-run, or set DRIVE_ZIP_PATH at the top of this "
        "cell to its exact path.")


def _find_csv_dir():
    """Return the folder that actually contains olist_orders_dataset.csv."""
    roots = []
    env_dir = os.environ.get("OLIST_DATA_PATH", "")   # set by the pipeline validator
    if env_dir:
        roots.append(env_dir)
    if LOCAL_DATA_DIR:
        roots.append(LOCAL_DATA_DIR)

    if ON_COLAB:
        extract_path = "/content/olist_data"
        # unzip only the first time; reuse the extracted CSVs afterwards
        if not glob.glob(f"{extract_path}/**/olist_orders_dataset.csv", recursive=True):
            zip_path = _colab_find_zip()
            os.makedirs(extract_path, exist_ok=True)
            print(f"Unzipping {os.path.basename(zip_path)} ...")
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(extract_path)
        roots.append(extract_path)
    else:
        # Local: search cwd (recursively) + a few common spots — never the whole
        # home dir (that recursive walk can be very slow). Set LOCAL_DATA_DIR if
        # your CSVs live elsewhere.
        roots += [os.getcwd(),
                  os.path.expanduser("~/Downloads"),
                  os.path.expanduser("~/Desktop"),
                  os.path.expanduser("~/olist")]

    for root in roots:
        if os.path.exists(os.path.join(root, "olist_orders_dataset.csv")):
            return root
        hits = glob.glob(os.path.join(root, "**", "olist_orders_dataset.csv"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])

    raise FileNotFoundError(
        "Olist CSVs not found. Set LOCAL_DATA_DIR (local) or DRIVE_ZIP_PATH (Colab) at "
        "the top of this cell.")


DATA_DIR = _find_csv_dir()
print("Data folder:", DATA_DIR)

# Build a file-based SQLite DB shared by pandas (loading) and jupysql (querying).
DB_PATH = os.environ.get("OLIST_DB_PATH") or (
    "/content/olist.db" if ON_COLAB else os.path.join(tempfile.gettempdir(), "olist.db"))

tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

conn = sqlite3.connect(DB_PATH)
for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")
conn.close()
print("\nDatabase ready.")

# On Colab, install jupysql so `%load_ext sql` loads it instead of the legacy
# ipython-sql (see header). Off Colab (local / pipeline validator) jupysql is
# already installed, so we skip the install and stay offline-safe.
if ON_COLAB:
    get_ipython().run_line_magic("pip", "install --quiet --upgrade jupysql")

get_ipython().run_line_magic("load_ext", "sql")

# Guard: if the legacy ipython-sql was already loaded earlier THIS session (e.g.
# an older cell ran first), the freshly installed jupysql cannot hot-swap in — a
# runtime restart is the only fix. jupysql exposes sql.connection.ConnectionManager;
# ipython-sql does not. Stop with a clear instruction instead of a later cryptic
# prettytable KeyError.
import sql.connection as _sqlconn
if not hasattr(_sqlconn, "ConnectionManager"):
    raise RuntimeError(
        "Legacy ipython-sql is active, not jupysql. On Colab: Runtime -> Restart session, "
        "then run THIS setup cell first (before any other cell). Locally: "
        "pip install --upgrade jupysql and restart the kernel."
    )

# Connect the %%sql magic to the SAME database file. autopandas=True is REQUIRED
# (see header). We connect with run_line_magic (not a literal `%sql` line) so the
# computed DB_PATH is interpolated correctly. Do NOT set SqlMagic.style.
get_ipython().run_line_magic("config", "SqlMagic.autopandas = True")
get_ipython().run_line_magic("config", "SqlMagic.feedback = 0")
get_ipython().run_line_magic("sql", f"sqlite:///{DB_PATH}")

# Verify (expected row counts — do not alter without re-running against data):
#   orders 99,441 | customers 99,441 | order_items 112,650 | order_payments 103,886
#   order_reviews 99,224 | products 32,951 | sellers 3,095 | product_category_translation 71


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data folder: /content/olist_data/phase-2-python-sql
Loaded orders: 99,441 rows
Loaded customers: 99,441 rows
Loaded order_items: 112,650 rows
Loaded order_payments: 103,886 rows
Loaded order_reviews: 99,224 rows
Loaded products: 32,951 rows
Loaded sellers: 3,095 rows
Loaded product_category_translation: 71 rows

Database ready.
The sql extension is already loaded. To reload it, use:
  %reload_ext sql


## Question 1 — The review score distribution

The `order_reviews` table holds one row per review, and `review_score` is a rating from 1 to 5. Write a `GROUP BY` query that returns each distinct `review_score` alongside how many reviews gave that score. Alias the count as `n`, and sort by `review_score` ascending (1 first, 5 last).

**Expected:** 5 rows — 1: 11,424 | 2: 3,151 | 3: 8,179 | 4: 19,142 | 5: 57,328

In [12]:
%%sql q1 <<
SELECT review_score, COUNT(*) AS n
FROM order_reviews
GROUP BY review_score
ORDER BY review_score ASC

In [13]:
# --- CHECK Q1 — do not edit ---
assert q1.shape[0] == 5, f"Q1: expected 5 score groups (1-5), got {q1.shape[0]}"
assert int(q1.iloc[0]['review_score']) == 1, "Q1: sort by review_score ascending — the first row should be score 1"
assert int(q1.iloc[0]['n']) == 11424, "Q1: expected 11,424 one-star reviews"
assert int(q1.iloc[4]['n']) == 57328, "Q1: expected 57,328 five-star reviews"
print("✅ Q1 correct")
q1  # show the result of your query

✅ Q1 correct


,review_score,n
0,1,11424
1,2,3151
2,3,8179
3,4,19142
4,5,57328


## Question 2 — Turn those counts into percentages

Counts are hard to compare across groups; percentages are not. Extend your Question 1 query so each row also shows what **share of all reviews** that score represents, rounded to 1 decimal place and aliased as `percentage`. Keep the count aliased as `n` and keep sorting by `review_score` ascending.

Hint: divide the group's count by the total number of reviews. You can get that total with a subquery in the `SELECT` list — `(SELECT COUNT(*) FROM order_reviews)` — and remember to multiply by `100.0` (not `100`) so SQLite does real division instead of truncating to zero.

**Expected:** 1: 11,424 (11.5%) | 2: 3,151 (3.2%) | 3: 8,179 (8.2%) | 4: 19,142 (19.3%) | 5: 57,328 (57.8%)

In [14]:
%%sql q2 <<
SELECT review_score, COUNT(*) AS n, COUNT(*) * 100.0 / (SELECT COUNT(*) FROM order_reviews) AS percentage
FROM order_reviews
GROUP BY review_score
ORDER BY review_score ASC

In [15]:
# --- CHECK Q2 — do not edit ---
assert q2.shape[0] == 5, f"Q2: expected 5 score groups (1-5), got {q2.shape[0]}"
assert 'percentage' in q2.columns, "Q2: alias the share column as 'percentage'"
assert round(float(q2.iloc[0]['percentage']), 1) == 11.5, "Q2: expected 11.5% one-star reviews (did you multiply by 100.0?)"
assert round(float(q2.iloc[2]['percentage']), 1) == 8.2, "Q2: expected 8.2% three-star reviews"
assert round(float(q2.iloc[4]['percentage']), 1) == 57.8, "Q2: expected 57.8% five-star reviews"
print("✅ Q2 correct")
q2  # show the result of your query

✅ Q2 correct


,review_score,n,percentage
0,1,11424,11.513344
1,2,3151,3.175643
2,3,8179,8.242965
3,4,19142,19.291704
4,5,57328,57.776344


## Question 3 — The headline number

Now collapse the whole table into a **single** row: what is the average `review_score` across every review in `order_reviews`? Round it to 2 decimal places and alias it as `overall_avg`. No `GROUP BY` is needed — an aggregate with no grouping treats the entire table as one group.

When it runs, compare it against your Question 2 output. The average looks healthy, but nearly 15% of reviews are 1 or 2 stars. That gap between the mean and the shape of the distribution is today's real lesson.

**Expected:** 4.09

In [23]:
%%sql q3 <<
SELECT ROUND(AVG(review_score), 2) AS overall_avg
FROM order_reviews;


In [19]:
# --- CHECK Q3 — do not edit ---
assert round(float(q3.iloc[0]['overall_avg']), 2) == 4.09, "Q3: expected an overall average review score of 4.09"
print("✅ Q3 correct")
q3  # show the result of your query

✅ Q3 correct


,overall_avg
0,4.09


## Question 4 — Total freight revenue

Unhappy reviews often trace back to delivery. Before you can look at that, you need to know how big shipping is as a line of business. The `order_items` table has one row per item sold, with a `freight_value` column holding the shipping charge for that item.

Add up `freight_value` across the whole table, round the total to 2 decimal places, and alias it as `total_freight_revenue`.

**Expected:** R$2,251,909.54

In [24]:
%%sql q4 <<
SELECT ROUND(SUM(freight_value), 2) AS total_freight_revenue
FROM order_items;

In [25]:
# --- CHECK Q4 — do not edit ---
assert round(float(q4.iloc[0]['total_freight_revenue']), 2) == 2251909.54, "Q4: expected total freight revenue of R$2,251,909.54"
print("✅ Q4 correct")
q4  # show the result of your query

✅ Q4 correct


,total_freight_revenue
0,2251909.54


## Question 5 — Only the high-volume scores (HAVING)

Go back to the grouped review-score query from Question 1, but this time keep only the scores that were given **more than 10,000 times**. The count only exists *after* the rows have been grouped, so this filter belongs in `HAVING`, not `WHERE`. Keep the count aliased as `n` and sort from the largest group down.

**Expected:** 3 rows — score 5 with 57,328 | score 4 with 19,142 | score 1 with 11,424

In [26]:
%%sql q5 <<
SELECT review_score, COUNT(*) AS n
FROM order_reviews
GROUP BY review_score
HAVING COUNT(*) > 10000
ORDER BY n DESC;

In [27]:
# --- CHECK Q5 — do not edit ---
assert q5.shape[0] == 3, f"Q5: expected 3 scores given more than 10,000 times, got {q5.shape[0]}"
assert int(q5.iloc[0]['review_score']) == 5, "Q5: top row should be score 5 — sort by the count descending"
assert int(q5.iloc[0]['n']) == 57328, "Q5: expected 57,328 five-star reviews"
assert int(q5.iloc[1]['n']) == 19142, "Q5: expected 19,142 four-star reviews in the second row"
assert int(q5.iloc[2]['review_score']) == 1, "Q5: the third row should be score 1"
assert int(q5.iloc[2]['n']) == 11424, "Q5: expected 11,424 one-star reviews"
print("✅ Q5 correct")
q5  # show the result of your query

✅ Q5 correct


,review_score,n
0,5,57328
1,4,19142
2,1,11424


## Question 6 — Discussion only (no code cell)

> 💬 **Think about this one, don't write SQL for it.** There is deliberately no answer cell and no check cell below.

**The question:** *Which customer state has the highest average review score?*

It sounds like a natural next step after Question 3 — you already know how to compute an average score, and on Wednesday you grouped customers by `customer_state`. So why can't you write it today?

Because the two pieces of information live in **two different tables**:

- `customer_state` is a column of the `customers` table.
- `review_score` is a column of the `order_reviews` table.

`GROUP BY` can only group columns that are already in the result you are querying. Every query you have written so far reads from exactly **one** table, so there is no single result set that contains both `customer_state` and `review_score` side by side. And the two tables are not even directly related — a review belongs to an `order_id`, and an order belongs to a `customer_id`, so you would have to travel `order_reviews → orders → customers` to connect them.

The tool that stitches tables together on a shared key is `JOIN`, and that is exactly what next week is about. Before you get there, be ready to answer these in discussion:

1. Which column links `order_reviews` to `orders`? Which column links `orders` to `customers`?
2. Once the three tables are combined, which column would you `GROUP BY`, and which one would you feed to `AVG()`?
3. `order_reviews` has 99,224 rows but `orders` has 99,441. What do you think happens to the orders that have no review when the tables are joined?

---

### Session summary

| Tool | What you used it for today |
|---|---|
| `GROUP BY review_score` | turned 99,224 individual reviews into 5 score groups |
| `COUNT(*)` | measured how big each group is |
| `* 100.0 / (SELECT COUNT(*) ...)` | converted counts into comparable percentages |
| `AVG()` with no `GROUP BY` | collapsed the whole table into one headline number (4.09) |
| `SUM()` | totalled freight revenue across `order_items` |
| `HAVING` | kept only the groups that passed a threshold |

**Coming up next week:** `JOIN` — combining tables so questions like Question 6 finally become answerable.